In [ ]:
# =============================================================================
# Cell 1：全局超参数与合成配置
# 仅需修改以下两个内容即可：
# 'WAKE_WORD_NAME': 'xiaoyuxiaoyu',  # 唤醒词标识名称（决定文件夹名与模型前缀）
# 'TEXT_VARIANTS': ['小雨', '小雨。', '小雨！'],
# =============================================================================
import json, os

CONFIG = {
    # 唤醒词标识名称（决定文件夹名与模型前缀）
    'WAKE_WORD_NAME': 'xiaoyuxiaoyu',

    # 音频时间窗口（毫秒）：短双字词设为 1500ms（必须能被 stride=3 整除）
    'CLIP_DURATION_MS': 1500,

    # 默认判定阈值（后续在 ESPHome 中可随时微调）
    'PROBABILITY_CUTOFF': 0.85,

    # 训练步数与负样本惩罚权重
    'TRAINING_STEPS': 10000,
    'NEGATIVE_CLASS_WEIGHT': 20,
    'EVAL_STEP_INTERVAL': 2000,

    # 训练文本变体（利用标点引入不同自然微停顿）
    'TEXT_VARIANTS': ['小雨小雨', '小雨，小雨。', '小雨！小雨！'],

    # 微软 Azure / Edge 神经拟真音色（融合男女声提高泛化度）
    'VOICES': [
        'zh-CN-XiaoxiaoNeural',  # 晓晓 (主力自然女声)
        'zh-CN-XiaoyiNeural',    # 晓伊 (标准播报女声)
        'zh-CN-YunxiNeural',     # 云希 (年轻男声)
        'zh-CN-YunjianNeural',   # 云健 (沉稳男声)
    ],

    # 多语速与微调音高（模拟真实环境不同说话习惯）
    'RATES': ['-15%', '-10%', '-5%', '+0%', '+5%', '+10%', '+15%'],
    'PITCHES': ['-5Hz', '+0Hz', '+5Hz'],
}

with open('/content/config.json', 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)

print(f"配置完成！唤醒词: {CONFIG['WAKE_WORD_NAME']}，音色数量: {len(CONFIG['VOICES'])} 个")

In [ ]:
# =============================================================================
# Cell 2：基础环境与核心依赖安装
# =============================================================================
%pip install 'datasets==2.21.0'
%pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

# 下载并以可编辑模式安装 microWakeWord
!test -d /content/microWakeWord || git clone https://github.com/kahrendt/microWakeWord /content/microWakeWord
%pip install -e /content/microWakeWord

# 安装 Edge-TTS、音频转换及特征生成依赖
%pip install edge-tts pydub soundfile librosa mmap-ninja

import importlib, site
importlib.invalidate_caches()
site.main()

import datasets
try:
    import microwakeword
    print('microwakeword 框架安装成功！')
except ModuleNotFoundError:
    raise RuntimeError('microWakeWord 安装失败，请检查上方日志。')

# 检查 datasets 版本是否生效
if datasets.__version__ != '2.21.0':
    print('\n*** 注意：旧版 datasets 仍残留在内存中 ***')
    print('*** 请点击 Colab 顶部菜单: 运行时 → 重启会话 (Restart session) ***')
    print('*** 重启后重新运行 Cell 1，然后跳过 Cell 2，直接运行 Cell 3 ***')
else:
    print('环境检查通过，可直接运行 Cell 3。')

In [ ]:
# =============================================================================
# Cell 3: 源码修复补丁
# =============================================================================
utils_path = '/content/microWakeWord/microwakeword/utils.py'

with open(utils_path, 'r', encoding='utf-8') as f:
    code = f.read()

old_assertion = '            assert spectrogram.shape[0] % stride == 0'
if old_assertion in code:
    new_logic = '''            remainder = spectrogram.shape[0] % stride
            if remainder != 0:
                spectrogram = spectrogram[:spectrogram.shape[0] - remainder]'''
    code = code.replace(old_assertion, new_logic)
    with open(utils_path, 'w', encoding='utf-8') as f:
        f.write(code)
    print('补丁修复成功：已将严格断言改为自动尾部截断。')
else:
    print('补丁已存在，无需重复修改。')

In [ ]:
# =============================================================================
# Cell 4: 批量生成高质量正样本
# =============================================================================
import os, json, asyncio, edge_tts
from pydub import AudioSegment

with open('/content/config.json', 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

os.makedirs('/content/generated_samples', exist_ok=True)
os.makedirs('/content/tmp_mp3', exist_ok=True)

sample_idx = 0

async def batch_generate():
    global sample_idx
    total_est = len(CONFIG['VOICES']) * len(CONFIG['RATES']) * len(CONFIG['PITCHES']) * len(CONFIG['TEXT_VARIANTS'])
    print(f"开始批量生成样本，预计总数: {total_est} 条...")

    for voice in CONFIG['VOICES']:
        for rate in CONFIG['RATES']:
            for pitch in CONFIG['PITCHES']:
                for text in CONFIG['TEXT_VARIANTS']:
                    mp3_file = f"/content/tmp_mp3/temp_{sample_idx}.mp3"
                    wav_file = f"/content/generated_samples/{sample_idx}.wav"

                    # 1. 调用 Edge-TTS 合成
                    comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
                    await comm.save(mp3_file)

                    # 2. 转为 16000Hz 16-bit 单声道 WAV 规范
                    seg = AudioSegment.from_file(mp3_file)
                    seg = seg.set_frame_rate(16000).set_channels(1).set_sample_width(2)
                    seg.export(wav_file, format='wav')

                    sample_idx += 1
                    if sample_idx % 250 == 0:
                        print(f"进度: 已处理并标准化 {sample_idx} 条样本...")

await batch_generate()
!rm -rf /content/tmp_mp3
print(f"样本生成完毕！总计输出 {sample_idx} 条标准 16kHz WAV 音频。")

In [ ]:
# =============================================================================
# Cell 5: 音频样本试听检验（已修复 Base64 嵌入与动态文件索引）
# =============================================================================
from IPython.display import Audio, display, HTML
import os

# 1. 获取实际生成的文件列表与数量
wav_files = sorted(
    [f for f in os.listdir('/content/generated_samples') if f.endswith('.wav')],
    key=lambda x: int(x.split('.')[0])
)
total_samples = len(wav_files)
print(f"当前目录共有 {total_samples} 个样本（编号 0.wav 到 {total_samples-1}.wav）\n")

# 2. 选取前、中、后 3 个代表性样本进行 Base64 嵌入试听
idx_start = 0
idx_mid = total_samples // 2
idx_end = total_samples - 1

display(HTML(f"<h4>试听 #{idx_start} 号样本 (晓晓女声 - 标准):</h4>"))
display(Audio(filename=f'/content/generated_samples/{idx_start}.wav', embed=True))

display(HTML(f"<h4>试听 #{idx_mid} 号样本 (不同语速/语调组合):</h4>"))
display(Audio(filename=f'/content/generated_samples/{idx_mid}.wav', embed=True))

display(HTML(f"<h4>试听 #{idx_end} 号样本 (男声音色):</h4>"))
display(Audio(filename=f'/content/generated_samples/{idx_end}.wav', embed=True))

In [ ]:
# =============================================================================
# Cell 6: 下载并标准化数据增强集 (RIRs / AudioSet / FMA)
# =============================================================================
import soundfile as sf, scipy, os, numpy as np
from pathlib import Path
from tqdm import tqdm
from datasets import load_dataset, Audio

# 1. MIT 房间脉冲响应（模拟房间混响反射）
rir_dir = './mit_rirs'
if not os.path.exists(rir_dir) or len(os.listdir(rir_dir)) == 0:
    os.makedirs(rir_dir, exist_ok=True)
    rirs = load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train')
    rirs = rirs.cast_column('audio', Audio(decode=False))
    for i, row in enumerate(tqdm(rirs, desc="处理 MIT 混响脉冲")):
        path = row['audio']['path']
        if path and os.path.exists(path):
            data, sr = sf.read(path)
            if len(data.shape) > 1: data = data[:, 0]
            if sr != 16000:
                import librosa
                data = librosa.resample(data, orig_sr=sr, target_sr=16000)
            scipy.io.wavfile.write(os.path.join(rir_dir, f'rir_{i}.wav'), 16000, (data * 32767).astype(np.int16))

# 2. AudioSet 环境背景音
if not os.path.exists('audioset'):
    os.mkdir('audioset')
    !wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !cd audioset && tar -xf bal_train09.tar

audioset_dir = './audioset_16k'
if not os.path.exists(audioset_dir) or len(os.listdir(audioset_dir)) == 0:
    os.makedirs(audioset_dir, exist_ok=True)
    for flac in tqdm(list(Path('audioset/audio').glob('**/*.flac')), desc="处理 AudioSet"):
        try:
            data, sr = sf.read(str(flac))
            if len(data.shape) > 1: data = data[:, 0]
            if sr != 16000:
                import librosa
                data = librosa.resample(data, orig_sr=sr, target_sr=16000)
            scipy.io.wavfile.write(os.path.join(audioset_dir, flac.stem + '.wav'), 16000, (data * 32767).astype(np.int16))
        except Exception: pass

# 3. FMA 音乐背景音
if not os.path.exists('fma') or not os.path.exists('fma/fma_small'):
    os.makedirs('fma', exist_ok=True)
    !wget -q -O fma/fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
    !cd fma && unzip -q fma_xs.zip

fma_dir = './fma_16k'
if not os.path.exists(fma_dir) or len(os.listdir(fma_dir)) == 0:
    os.makedirs(fma_dir, exist_ok=True)
    from pydub import AudioSegment
    for mp3 in tqdm(list(Path('fma/fma_small').glob('**/*.mp3')), desc="处理 FMA 背景音"):
        try:
            seg = AudioSegment.from_mp3(str(mp3)).set_frame_rate(16000).set_channels(1)
            samples = np.array(seg.get_array_of_samples(), dtype=np.int16)
            scipy.io.wavfile.write(os.path.join(fma_dir, mp3.stem + '.wav'), 16000, samples)
        except Exception: pass

print('数据增强集准备就绪！')

In [ ]:
# =============================================================================
# Cell 7: 声学特征增强与 mmap 构建
# =============================================================================
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
import os

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

out_features_dir = 'generated_augmented_features'
for split in ['training', 'validation', 'testing']:
    out_dir = os.path.join(out_features_dir, split)
    os.makedirs(out_dir, exist_ok=True)

    split_name = {'training': 'train', 'validation': 'validation', 'testing': 'test'}[split]
    repetition = 2 if split == 'training' else 1
    slide = 1 if split == 'testing' else 10

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=slide,
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True,
    )

print('声学特征工程切片完毕！')

In [ ]:
# =============================================================================
# Cell 8: 拉取预制负样本集 (日常闲聊、聚会对话、静音)
# =============================================================================
import os

neg_dir = './negative_datasets'
if not os.path.exists(neg_dir):
    os.mkdir(neg_dir)
    link_base = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    files = ['dinner_party.zip', 'no_speech.zip', 'speech.zip']
    for f in files:
        zip_path = f'negative_datasets/{f}'
        !wget -q -O {zip_path} {link_base + f}
        !unzip -q {zip_path} -d {neg_dir}

print('负样本数据集已准备完毕！')

In [ ]:
# =============================================================================
# Cell 9: 生成训练配置文件 (training_parameters.yaml)
# =============================================================================
import json, yaml

with open('/content/config.json', 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

train_cfg = {
    'window_step_ms': 10,
    'train_dir': f"trained_models/{CONFIG['WAKE_WORD_NAME']}",
    'features': [
        {
            'features_dir': 'generated_augmented_features',
            'sampling_weight': 2.0,
            'penalty_weight': 1.0,
            'truth': True,
            'truncation_strategy': 'truncate_start',
            'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/speech',
            'sampling_weight': 10.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/dinner_party',
            'sampling_weight': 10.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/no_speech',
            'sampling_weight': 5.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
    ],
    'training_steps': [CONFIG['TRAINING_STEPS']],
    'positive_class_weight': [1],
    'negative_class_weight': [CONFIG['NEGATIVE_CLASS_WEIGHT']],
    'learning_rates': [0.001],
    'batch_size': 128,
    'time_mask_max_size': [0],
    'time_mask_count': [0],
    'freq_mask_max_size': [0],
    'freq_mask_count': [0],
    'eval_step_interval': CONFIG['EVAL_STEP_INTERVAL'],
    'clip_duration_ms': CONFIG['CLIP_DURATION_MS'],
    'target_minimization': 0.9,
    'minimization_metric': None,
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(train_cfg, f)

print(f"配置文件已生成: trained_models/{CONFIG['WAKE_WORD_NAME']} (clip_duration: {CONFIG['CLIP_DURATION_MS']}ms)")

In [ ]:
# =============================================================================
# Cell 10: 启动训练与量化导出 (无需人工干预)
# =============================================================================
!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# =============================================================================
# Cell 11: 生成 ESPHome 配置并下载
# =============================================================================
from google.colab import files
import os, json, shutil

# 1. 自动读取 Cell 1 保存的全局配置
with open('/content/config.json', 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

wake_name = CONFIG['WAKE_WORD_NAME']
wake_display = CONFIG['TEXT_VARIANTS'][0].rstrip('。！!?,. ')
cutoff = CONFIG.get('PROBABILITY_CUTOFF', 0.85)

model_dir = f"trained_models/{wake_name}"
tflite_name = f"{wake_name}_stream.tflite"
json_name = f"{wake_name}.json"

# 2. 自动搜索训练导出的 .tflite 模型并重命名
found_model = False
for root, dirs, fnames in os.walk(model_dir):
    for f in fnames:
        if f.endswith('.tflite') and not f.endswith(tflite_name):
            src = os.path.join(root, f)
            dst = os.path.join(model_dir, tflite_name)
            shutil.copyfile(src, dst)
            found_model = True
            print(f"已定位模型文件: {f} -> 重命名为: {tflite_name}")
            break
    if found_model:
        break

# 3. 动态组装 ESPHome microWakeWord 描述清单
manifest = {
    "type": "micro",
    "wake_word": wake_display,
    "author": "Allen",
    "model": tflite_name,
    "trained_languages": ["zh"],
    "version": 2,
    "micro": {
        "probability_cutoff": cutoff,
        "sliding_window_size": 3,
        "feature_step_size": 10,
        "tensor_arena_size": 28000,
        "minimum_esphome_version": "2024.7"
    }
}

json_path = os.path.join(model_dir, json_name)
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"\n成功生成描述清单: {json_name} (唤醒词: {wake_display}, 阈值: {cutoff})")
print("正在弹出浏览器下载窗口...")

# 4. 自动触发下载
files.download(os.path.join(model_dir, tflite_name))
files.download(json_path)
print(f"下载已触发！请查收 {tflite_name} 和 {json_name}。")